In [ ]:
import feedparser
from openai import OpenAI
import datetime
from dateutil import parser as dateparser
import json
import os
from bs4 import BeautifulSoup # For cleaning HTML from summaries
from dotenv import load_dotenv

# ------------- CONFIG ----------------

# Load environment variables from .env file (must be in the same directory)
load_dotenv()
   
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
  
if not OPENAI_API_KEY:
	raise ValueError("OPENAI_API_KEY not found. Make sure it's set in your .env file.")

# Use the modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Using gpt-4o-mini: faster, more capable, and cost-effective for this task.
LLM_MODEL = "gpt-4o-mini"

# --- NEW, CURATED RSS FEEDS ---
RSS_FEEDS = {
    # NEP (New Economic Papers from RePEc) - Core Economics
    "NEP-FOR": "https://nep.repec.org/nep-for.rss.xml",   # **Forecasting** (Most Important)
    "NEP-ETS": "https://nep.repec.org/nep-ets.rss.xml",   # Econometric Time Series
    "NEP-BIG": "https://nep.repec.org/nep-big.rss.xml",   # Big Data
    "NEP-CMP": "https://nep.repec.org/nep-cmp.rss.xml",   # Computational Economics
    "NEP-MAC": "https://nep.repec.org/nep-mac.rss.xml",   # Macroeconomics
    "NEP-MON": "https://nep.repec.org/nep-mon.rss.xml",   # Monetary Economics
    
    # NEP - Finance-Focused
    "NEP-FMK": "https://nep.repec.org/nep-fmk.rss.xml",   # Financial Markets
    "NEP-DER": "https://nep.repec.org/nep-der.rss.xml",   # Derivatives
    "NEP-RMG": "https://nep.repec.org/nep-rmg.rss.xml",   # Risk Management
    "NEP-MST": "https://nep.repec.org/nep-mst.rss.xml",   # Market Microstructure

    # NEP - Energy & Resource Focused
    "NEP-ENE": "https://nep.repec.org/nep-ene.rss.xml",   # Energy Economics
    "NEP-RES": "https://nep.repec.org/nep-res.rss.xml",   # Resource Economics

    # ArXiv - Core Quantitative & CS Fields
    "arXiv-econEM": "http://export.arxiv.org/rss/econ.EM",     # Econometrics
    "arXiv-statML": "http://export.arxiv.org/rss/stat.ML",     # Statistics - Machine Learning
    "arXiv-csLG": "http://export.arxiv.org/rss/cs.LG",         # Computer Science - Machine Learning
    "arXiv-qFIN-ST": "http://export.arxiv.org/rss/q-fin.ST",   # Quantitative Finance - Statistical Methods
    "arXiv-qFIN-CP": "http://export.arxiv.org/rss/q-fin.CP",   # Quantitative Finance - Computational Finance

    # SSRN eJournals - High Volume Preprints
    "SSRN-ECMET": "https://papers.ssrn.com/sol3/JELJOUR_Results.cfm?form_name=journalBrowse&journal_id=383248&SortOrder=desc&ctag=RSS-2.0", # Econometric & Statistical Methods
    "SSRN-FIN-ECMET": "https://papers.ssrn.com/sol3/JELJOUR_Results.cfm?form_name=journalBrowse&journal_id=658939&SortOrder=desc&ctag=RSS-2.0", # Financial Econometrics
    "SSRN-ENERGY": "https://papers.ssrn.com/sol3/JELJOUR_Results.cfm?form_name=journalBrowse&journal_id=956426&SortOrder=desc&ctag=RSS-2.0", # Energy Economics
    
    # Energy-Focused Agencies & Orgs
    "EIA_Analysis": "https://www.eia.gov/rss/todayinenergy.xml",            # US Energy Information Administration
    "IEA_Reports": "https://www.iea.org/rss/reports",                       # International Energy Agency
    "OPEC_MOMR": "https://www.opec.org/opec_web/en/press_room/332.htm/rss",  # OPEC Monthly Oil Market Report News

    # High-Signal Institutions (Central Banks & Academic Orgs)
    "FEDS_Papers": "https://www.federalreserve.gov/feeds/feds_working_papers.xml", # US Federal Reserve Board
    "ECB_Papers": "https://www.ecb.europa.eu/pub/rss/wpp.html",                   # European Central Bank
    "IMF_Papers": "https://www.imf.org/en/Publications/RSS?language=eng&series=IMF%20Working%20Papers", # International Monetary Fund
    "BIS_Papers": "https://www.bis.org/doclist/rss_wp.xml",                       # Bank for International Settlements
    "BOE_Papers": "https://www.bankofengland.co.uk/rss/workingpapers",            # Bank of England
    "OFR_Papers": "https://www.financialresearch.gov/working-papers/feed/",       # US Office of Financial Research
    "NBER_Papers": "https://www.nber.org/new.xml",                                # National Bureau of Economic Research (NBER)
    "CEPR_Papers": "https://cepr.org/taxonomy/term/123/feed",                     # Centre for Economic Policy Research (CEPR)

    # --- NEW: Policy & Economic Analysis Think Tanks ---
    "Bruegel_Pubs": "https://www.bruegel.org/publications/feed",                  # Bruegel - Leading European economic policy think tank
    "PIIE_Pubs": "https://www.piie.com/publications/rss.xml",                     # Peterson Institute for International Economics
    "CBO_Pubs": "https://www.cbo.gov/publications/all/rss.xml",                   # US Congressional Budget Office - Produces key US economic forecasts
    "WorldBank_Econ": "https://blogs.worldbank.org/rss/category/all/all/term/1831", # World Bank - Economic Prospects blog/reports
    "OECD_Econ": "https://www.oecd.org/economy/rss.xml",                          # OECD - Economic department reports and outlooks
    "CSIS_Energy": "https://www.csis.org/programs/energy-security-and-climate-change-program/rss.xml", # CSIS - Energy Security & Climate Change
    "RFF_Pubs": "https://www.rff.org/feed/",                                      # Resources for the Future - Energy & Environmental Economics
    "Brookings_Econ": "https://www.brookings.edu/program/economic-studies/feed/", # Brookings Institution - Economic Studies
}

SCORE_THRESHOLD = 7.0  # Keep only papers scoring above this
DAYS_BACK = 7          # Look back this many days

# Using a relative path is more portable and a best practice.
# This will create a 'newsletters' folder where the script is run.
OUTPUT_DIR = "newsletters"

# ------------- HELPER FUNCTIONS ----------------

def _get_authors(entry):
    """
    Normalizes author information from a feed entry.
    Handles both 'authors' list (preferred) and 'author' string.
    """
    if hasattr(entry, 'authors') and entry.authors:
        # Typically a list of dicts with a 'name' key
        return ', '.join(author['name'] for author in entry.authors if 'name' in author)
    if hasattr(entry, 'author'):
        return entry.author
    return "Unknown"

# ------------- CORE FUNCTIONS ----------------

def fetch_recent_papers():
    """
    Fetch recent papers (last DAYS_BACK) from RSS feeds.
    Includes robust date parsing and error handling.
    """
    print(f"Fetching papers from {len(RSS_FEEDS)} sources, looking back {DAYS_BACK} days...")
    cutoff_date = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=DAYS_BACK)
    papers = []

    for source, url in RSS_FEEDS.items():
        print(f"  - Processing {source} from {url}")
        try:
            feed = feedparser.parse(url)
            if feed.bozo:
                print(f"    Warning: Malformed feed for {source}. Error: {feed.bozo_exception}")

            found_in_feed = 0
            for entry in feed.entries:
                pub_date_str = getattr(entry, "published", getattr(entry, "updated", None))
                if not pub_date_str:
                    continue

                try:
                    pub_date = dateparser.parse(pub_date_str)
                except dateparser.ParserError:
                    continue

                if pub_date.tzinfo is None:
                    pub_date = pub_date.replace(tzinfo=datetime.timezone.utc)

                if pub_date < cutoff_date:
                    continue

                summary = getattr(entry, "summary", "")
                if "<" in summary and ">" in summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(separator=' ', strip=True)

                papers.append({
                    "title": entry.title,
                    "link": entry.link,
                    "summary": summary,
                    "authors": _get_authors(entry),
                    "source": source,
                    "date": pub_date.strftime("%Y-%m-%d")
                })
                found_in_feed += 1
            print(f"    Found {found_in_feed} recent papers in {source}.")

        except Exception as e:
            print(f"    Error fetching or parsing feed {source}: {e}")
            continue
    return papers

def score_and_summarize(paper):
    """
    Sends abstract to LLM for summary and scoring based on novel economic forecasting.
    Uses the modern OpenAI client and robust error handling.
    """
    # --- NEW, MORE DETAILED PROMPT ---
    prompt = f"""
You are a highly specialized expert curator for "ets4 Weekly (Economic Time Series Forecasting Weekly)," a newsletter focused on cutting-edge and novel methods for forecasting economic time series. Your audience consists of researchers and practitioners who need to stay on the frontier of this field.

Your task is to meticulously evaluate the provided research paper.

Here is the paper's information:

Title: {paper['title']}
Authors: {paper['authors']}
Source: {paper['source']}
Date: {paper['date']}
Abstract: {paper['summary']}

Please perform the following three steps:

1.  **Summarize the paper:** Provide a concise summary (2-3 sentences) focusing on its objective, methods, and findings as they relate to time series forecasting.

2.  **Assign a quality score:** Give a numerical score from 1 to 10 based on its methodological novelty, rigor, and potential impact within its own field. This score reflects the paper's quality, independent of its direct relevance to economics.

3.  **Categorize and Justify:** Classify the paper into one of three categories. If you choose "Paper of Interest," you must provide a justification.
    *   **Directly Relevant:** The paper is a strong fit for the newsletter's core economic focus.
    *   **Paper of Interest:** The paper is NOT about economics, but its novel time series forecasting methods are highly innovative and could be adapted for economic applications.
    *   **Not Relevant:** The paper is out of scope.

**Evaluation Criteria for "Directly Relevant" Papers:**

A paper is considered **Directly Relevant** if it is focused on economics and involves:
*   Introducing a new forecasting model or method for economic time series.
*   Applying forecasting to economic policy, finance, or business decisions (e.g., trading strategies, inflation, business cycles).
*   Methods for forecast evaluation, including hypothesis testing of predictive ability and forecast comparison.
*   Techniques for forecast combination or model averaging.
*   Nowcasting or high-frequency forecasting of economic indicators or financial prices.
*   Using novel or alternative data for economic forecasting.
*   Introducing a novel business cycle indicator, economic condition index, or uncertainty measure.
*   Insightful policy briefs or analyses focused on economic projections.

A paper is **Not Relevant** if its primary focus is on non-forecasting topics like causal inference, purely theoretical econometrics without predictive application, or descriptive analysis.

**Scoring & Categorization Rubric:**
*   **Score 1-3:** Low quality or not a forecasting paper. -> **Category: "Not Relevant"**
*   **Score 4-6:** Uses standard methods, is incremental, or is a niche application with limited generalizability. -> **Category: "Not Relevant"**
*   **Score 7-8:** A high-quality paper making a clear contribution.
    *   If it meets the "Directly Relevant" criteria -> **Category: "Directly Relevant"**.
    *   If it's non-economic but has a novel, adaptable method -> **Category: "Paper of Interest"**.
*   **Score 9-10:** A groundbreaking paper with major methodological or empirical impact.
    *   If it meets the "Directly Relevant" criteria -> **Category: "Directly Relevant"**.
    *   If it's non-economic but has a truly revolutionary, adaptable method -> **Category: "Paper of Interest"**.

Return your response strictly in JSON format as shown below. For "Paper of Interest", the `adaptability_reason` field is mandatory. For all other categories, this field must be `null`.
{{
  "summary": "Your 2-3 sentence summary here.",
  "score": number,
  "category": "Directly Relevant" | "Paper of Interest" | "Not Relevant",
  "adaptability_reason": "A single sentence explaining why this non-economic paper's methods can be adapted to economics." | null
}}
"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        data = json.loads(content)
        if "summary" not in data or "score" not in data:
            raise ValueError("LLM response missing 'summary' or 'score' key.")
        if not isinstance(data["score"], (int, float)):
            raise ValueError("LLM score is not a number.")
        
        data["score"] = float(data["score"])
        return data
    except Exception as e:
        print(f"Error during LLM call for paper '{paper['title']}': {e}")
        return {"summary": f"Error during processing: {e}", "score": 0.0}

def build_markdown(papers, filename):
    """
    Create markdown file with shortlisted papers.
    Ensures output directory exists and provides a clear header.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    lines = [
        f"# ets4 Weekly – {datetime.date.today().strftime('%Y-%m-%d')}\n",
        "A curated list of recent papers on novel economic forecasting methods.\n",
        "## Candidate Papers\n"
    ]
    if not papers:
        lines.append("No papers met the criteria this week. Check back soon!\n")
    else:
        papers.sort(key=lambda p: p['score'], reverse=True)
        for p in papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Relevance Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}\n")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✅ Markdown file saved: {filename}")

# ------------- MAIN ----------------

if __name__ == "__main__":
    print("Starting Economic Forecasting Newsletter Pipeline...")
    
    all_papers = fetch_recent_papers()
    print(f"Found {len(all_papers)} total recent papers.")

    unique_papers = []
    seen_links = set()
    for paper in all_papers:
        if paper['link'] not in seen_links:
            unique_papers.append(paper)
            seen_links.add(paper['link'])
    
    if len(all_papers) > len(unique_papers):
        print(f"Removed {len(all_papers) - len(unique_papers)} duplicates. Processing {len(unique_papers)} unique papers.")

    shortlisted = []
    for i, paper in enumerate(unique_papers, 1):
        print(f"Processing paper {i}/{len(unique_papers)}: '{paper['title'][:70]}...'")
        result = score_and_summarize(paper)
        paper.update(result)
        if paper["score"] >= SCORE_THRESHOLD:
            shortlisted.append(paper)
            print(f"  -> Shortlisted! Score: {paper['score']:.1f}/10")
        else:
            print(f"  -> Skipped. Score: {paper['score']:.1f}/10 (Threshold: {SCORE_THRESHOLD})")

    today_str = datetime.date.today().strftime("%Y-%m-%d")
    output_filename = os.path.join(OUTPUT_DIR, f"ets4_weekly_{today_str}.md")
    build_markdown(shortlisted, output_filename)
    
    print("\n--- Pipeline Finished ---")
    print(f"Total unique papers processed by LLM: {len(unique_papers)}")
    print(f"Shortlisted {len(shortlisted)} papers (score ≥ {SCORE_THRESHOLD}).")

Starting Economic Forecasting Newsletter Pipeline...
Fetching papers from 39 sources, looking back 7 days...
  - Processing NEP-FOR from https://nep.repec.org/nep-for.rdf
    Found 0 recent papers in NEP-FOR.
  - Processing NEP-ETS from https://nep.repec.org/nep-ets.rdf
    Found 0 recent papers in NEP-ETS.
  - Processing NEP-BIG from https://nep.repec.org/nep-big.rdf
    Found 0 recent papers in NEP-BIG.
  - Processing NEP-CMP from https://nep.repec.org/nep-cmp.rdf
    Found 0 recent papers in NEP-CMP.
  - Processing NEP-MAC from https://nep.repec.org/nep-mac.rdf
    Found 0 recent papers in NEP-MAC.
  - Processing NEP-MON from https://nep.repec.org/nep-mon.rdf
    Found 0 recent papers in NEP-MON.
  - Processing NEP-FMK from https://nep.repec.org/nep-fmk.rdf
    Found 0 recent papers in NEP-FMK.
  - Processing NEP-DER from https://nep.repec.org/nep-der.rdf
    Found 0 recent papers in NEP-DER.
  - Processing NEP-RMG from https://nep.repec.org/nep-rmg.rdf
    Found 0 recent papers in 

<python-site-packages>/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


    Found 2 recent papers in EIA_Analysis.
  - Processing IEA_Reports from https://www.iea.org/rss/reports
    Found 0 recent papers in IEA_Reports.
  - Processing OPEC_MOMR from https://www.opec.org/opec_web/en/press_room/332.htm/rss
    Found 0 recent papers in OPEC_MOMR.
  - Processing FEDS_Papers from https://www.federalreserve.gov/feeds/feds_working_papers.xml
    Found 0 recent papers in FEDS_Papers.
  - Processing ECB_Papers from https://www.ecb.europa.eu/pub/rss/wpp.html
    Found 0 recent papers in ECB_Papers.
  - Processing IMF_Papers from https://www.imf.org/en/Publications/RSS?language=eng&series=IMF%20Working%20Papers
    Found 6 recent papers in IMF_Papers.
  - Processing BIS_Papers from https://www.bis.org/doclist/rss_wp.xml
    Found 0 recent papers in BIS_Papers.
  - Processing BOE_Papers from https://www.bankofengland.co.uk/rss/workingpapers
    Found 0 recent papers in BOE_Papers.
  - Processing OFR_Papers from https://www.financialresearch.gov/working-papers/feed/
 